# Time Series Analysis Models for S&P 500

This notebook demonstrates two common time series analysis techniques for the S&P 500 index:

1. ARIMA (Autoregressive Integrated Moving Average) for forecasting future index returns
2. Exponential Smoothing for volatility estimation and forecasting

Both techniques are foundational in quantitative finance and algorithmic trading.

## Setup and Data Collection

First, let's import the necessary libraries and fetch S&P 500 historical data.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from pandas_datareader import data as pdr
import yfinance as yf
yf.pdr_override()

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 12

AttributeError: module 'yfinance' has no attribute 'pdr_override'

In [ ]:
# Get S&P 500 historical data (last 3 years)
sp500 = pdr.get_data_yahoo('^GSPC', start='2022-01-01')

# Display the first few rows
print(f"Data period: {sp500.index.min().date()} to {sp500.index.max().date()}")
print(f"Number of trading days: {len(sp500)}")
sp500.head()

AttributeError: module 'pandas_datareader.data' has no attribute 'override'

Let's visualize the S&P 500 price history:

In [ ]:
plt.figure(figsize=(14, 7))
plt.plot(sp500['Close'])
plt.title('S&P 500 Index (^GSPC)')
plt.xlabel('Date')
plt.ylabel('Price')
plt.grid(True)
plt.tight_layout()
plt.show()

## 1. ARIMA Model for S&P 500 Index Forecasting

ARIMA (Autoregressive Integrated Moving Average) models are widely used for time series forecasting. Here, we'll build an ARIMA model to forecast future returns of the S&P 500 index.

First, let's calculate daily returns and visualize them:

In [ ]:
# Calculate daily returns
sp500['Returns'] = sp500['Close'].pct_change()
returns = sp500['Returns'].dropna()

# Plot returns
plt.figure(figsize=(14, 7))
plt.plot(returns)
plt.title('S&P 500 Daily Returns')
plt.xlabel('Date')
plt.ylabel('Return')
plt.grid(True)
plt.tight_layout()
plt.show()

Let's analyze the statistical properties of these returns:

In [ ]:
# Calculate summary statistics for returns
print("Statistical properties of daily returns:")
print(f"Mean: {returns.mean():.6f}")
print(f"Standard Deviation: {returns.std():.6f}")
print(f"Minimum: {returns.min():.6f}")
print(f"Maximum: {returns.max():.6f}")
print(f"Skewness: {returns.skew():.6f}")
print(f"Kurtosis: {returns.kurtosis():.6f}")

Now, let's fit an ARIMA model to the return series. 

We'll use an ARIMA(5,1,2) model, which means:
- 5 autoregressive terms (AR)
- 1st order differencing (I)
- 2 moving average terms (MA)

In [ ]:
# Fit ARIMA model
model = ARIMA(returns, order=(5, 1, 2))
model_fit = model.fit()

# Display model summary
print(model_fit.summary())

Let's plot the model's diagnostics to assess its fit:

In [ ]:
# Plot model diagnostics
model_fit.plot_diagnostics(figsize=(14, 10))
plt.tight_layout()
plt.show()

Now, let's forecast the next 10 trading days:

In [ ]:
# Forecast next 10 days
forecast = model_fit.forecast(steps=10)
forecast_df = pd.DataFrame({
    'Forecasted Return': forecast,
    'Lower CI': forecast - 1.96 * model_fit.params['sigma2'] ** 0.5,
    'Upper CI': forecast + 1.96 * model_fit.params['sigma2'] ** 0.5
})

print("Forecasted S&P 500 returns for next 10 trading days:")
forecast_df

In [ ]:
# Plot forecasted returns
plt.figure(figsize=(14, 7))

# Plot historical returns
plt.plot(returns[-30:], label='Historical Returns')

# Create forecast index starting from the last date in the historical data
last_date = returns.index[-1]
forecast_idx = pd.date_range(start=last_date + pd.Timedelta(days=1), periods=10, freq='B')
forecast_series = pd.Series(forecast, index=forecast_idx)

# Plot forecast
plt.plot(forecast_series, color='red', label='Forecasted Returns')
plt.title('S&P 500 Returns Forecast (10 days)')
plt.xlabel('Date')
plt.ylabel('Return')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## 2. Exponential Smoothing for Volatility Estimation

Exponential smoothing is another popular technique for time series analysis. Here, we'll use it to model and forecast the volatility of the S&P 500.

First, let's calculate the rolling volatility:

In [ ]:
# Calculate rolling volatility (20-day window)
sp500['Volatility'] = sp500['Returns'].rolling(window=20).std() * np.sqrt(252)

# Plot volatility
plt.figure(figsize=(14, 7))
plt.plot(sp500['Volatility'].dropna())
plt.title('S&P 500 20-Day Rolling Volatility (Annualized)')
plt.xlabel('Date')
plt.ylabel('Volatility')
plt.grid(True)
plt.tight_layout()
plt.show()

Now, let's apply exponential smoothing to the volatility series:

In [ ]:
# Apply exponential smoothing to volatility
volatility_series = sp500['Volatility'].dropna()
model = ExponentialSmoothing(volatility_series, trend='add', seasonal=None)
fit = model.fit()

# Plot actual vs. fitted values
plt.figure(figsize=(14, 7))
plt.plot(volatility_series, label='Actual Volatility')
plt.plot(fit.fittedvalues, color='red', label='Fitted Values')
plt.title('S&P 500 Volatility: Actual vs. Exponential Smoothing')
plt.xlabel('Date')
plt.ylabel('Volatility')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

Now, let's forecast future volatility:

In [ ]:
# Forecast volatility for next 20 trading days
volatility_forecast = fit.forecast(20)

# Create a date range for the forecast period
last_date = volatility_series.index[-1]
forecast_idx = pd.date_range(start=last_date + pd.Timedelta(days=1), periods=20, freq='B')
volatility_forecast.index = forecast_idx

# Plot forecast
plt.figure(figsize=(14, 7))
plt.plot(volatility_series[-60:], label='Historical Volatility')
plt.plot(volatility_forecast, color='red', label='Forecasted Volatility')
plt.title('S&P 500 Volatility Forecast (20 days)')
plt.xlabel('Date')
plt.ylabel('Volatility')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Display forecasted volatility
print("Volatility forecast for next 20 trading days (annualized):")
pd.DataFrame({
    'Date': volatility_forecast.index.date,
    'Forecasted Volatility': volatility_forecast.values
})

## Algorithmic Trading Applications

Here are some ways these time series models can be applied in algorithmic trading:

1. **ARIMA Return Forecasts**:
   - Generate buy/sell signals based on the predicted direction of returns
   - Use forecasted returns to optimize portfolio allocation
   - Combine with other signals for enhanced trading strategies

2. **Volatility Forecasts**:
   - Size positions inversely proportional to expected volatility
   - Trade volatility directly through VIX derivatives
   - Set dynamic stop-loss levels based on predicted volatility
   - Adjust options strategies based on forecasted volatility
   
Both models provide valuable information for risk management and trade timing in algorithmic strategies.